# Phase 6: Advanced Models — CatBoost & TabM

Notebook độc lập, bổ sung **hai phương pháp hiện đại** cạnh các baseline ở Phase 2,
đánh giá bằng **đúng giao thức 5-fold Stratified CV** để so sánh công bằng, rồi xuất submission.

## Hai phương pháp

**CatBoost** *(Gradient Boosting trên cây)* — chống overfit & rò rỉ target bằng **ordered boosting**
và **ordered target statistics**; dùng **oblivious trees** (mỗi tầng cùng một điều kiện cắt) như một
dạng regularization. Rất mạnh trên dữ liệu nhỏ + nhiều biến phân loại.
Paper (NeurIPS 2018): https://arxiv.org/abs/1706.09516

**TabM** *(Deep tabular)* — một MLP dùng **parameter-efficient ensembling** (BatchEnsemble):
toàn bộ `k` thành viên dùng chung một ma trận trọng số `W`, mỗi thành viên chỉ thêm hai vector
rank-1 `r, s` để điều biến → một mạng mô phỏng `k` MLP, lấy trung bình `k` dự đoán.
Ở đây TabM được **cài đặt lại từ đầu bằng PyTorch** (theo tinh thần notebook 04).
Paper (ICLR 2025): https://arxiv.org/abs/2410.24210 · Code: https://github.com/yandex-research/tabm


In [ ]:
# === Imports & cấu hình ===
# Nếu thiếu thư viện, bỏ comment dòng dưới:
# %pip install catboost torch scikit-learn matplotlib

import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                             classification_report, confusion_matrix, RocCurveDisplay)

import torch
import torch.nn as nn

from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

## Step 0 — Nạp dữ liệu đã làm sạch

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/processed")

train = pd.read_csv(DATA_DIR / "train_cleaned.csv")
test  = pd.read_csv(DATA_DIR / "test_cleaned.csv")

TARGET = "Class/ASD"
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_test = test[X.columns]          # ép đúng thứ tự cột như train

assert list(X.columns) == list(X_test.columns), "Cột train/test không khớp!"
print(f"train: {X.shape} | test: {X_test.shape}")
print("Phân bố lớp:", y.value_counts().to_dict())

# Cùng một CV với Phase 2 để so sánh công bằng
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

X_np, y_np = X.values.astype(np.float32), y.values.astype(np.int64)
leaderboard = []   # sẽ gom kết quả các model vào đây

## 1. CatBoost — 5-fold Stratified CV

Đánh giá CatBoost trên cùng dữ liệu đã encode (so sánh apples-to-apples với baseline).
`auto_class_weights="Balanced"` xử lý mất cân bằng lớp.

In [ ]:
def cb_params():
    return dict(iterations=400, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
                loss_function="Logloss", eval_metric="AUC",
                auto_class_weights="Balanced", random_seed=SEED, verbose=False)

cb_auc, cb_f1, cb_acc = [], [], []
t0 = time.time()
for fold, (tr, va) in enumerate(cv.split(X_np, y_np), 1):
    model = CatBoostClassifier(**cb_params())
    model.fit(X_np[tr], y_np[tr])
    p = model.predict_proba(X_np[va])[:, 1]
    pred = (p >= 0.5).astype(int)
    cb_auc.append(roc_auc_score(y_np[va], p))
    cb_f1.append(f1_score(y_np[va], pred))
    cb_acc.append(accuracy_score(y_np[va], pred))
    print(f"  fold {fold}: AUC={cb_auc[-1]:.4f}")

print(f"\nCatBoost | AUC {np.mean(cb_auc):.4f} ± {np.std(cb_auc):.4f} "
      f"| F1 {np.mean(cb_f1):.4f} | Acc {np.mean(cb_acc):.4f} | {time.time()-t0:.1f}s")

leaderboard.append({"Model": "CatBoost",
                    "Accuracy": np.mean(cb_acc), "Accuracy-Std": np.std(cb_acc),
                    "F1-Score": np.mean(cb_f1), "F1-Std": np.std(cb_f1),
                    "ROC-AUC": np.mean(cb_auc), "ROC-AUC-Std": np.std(cb_auc)})

In [ ]:
# Fit trên toàn bộ train để lấy feature importance + ROC + dùng cho submission
cb_final = CatBoostClassifier(**cb_params()).fit(X_np, y_np)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
imp = pd.Series(cb_final.feature_importances_, index=X.columns).sort_values()
imp.tail(15).plot.barh(ax=ax[0], color="#2b8cbe")
ax[0].set_title("CatBoost — Top 15 Feature Importance")

RocCurveDisplay.from_predictions(y_np, cb_final.predict_proba(X_np)[:, 1], ax=ax[1], name="CatBoost")
ax[1].plot([0, 1], [0, 1], "k--", lw=1); ax[1].set_title("CatBoost ROC (train)")
plt.tight_layout(); plt.show()

### (Tùy chọn) SHAP cho CatBoost — nối với Phase 5 (Explainable AI)

Chạy nếu đã cài `shap` (`%pip install shap`). Bỏ qua được nếu chỉ cần leaderboard.

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(cb_final)
    sv = explainer.shap_values(X_np)
    shap.summary_plot(sv, X, plot_type="bar", show=True)
except Exception as e:
    print("Bỏ qua SHAP:", e)

## 2. TabM — cài đặt từ đầu (PyTorch)

`BatchEnsembleLinear`: một trọng số `W` dùng chung + cặp adapter rank-1 `r, s` cho từng thành viên.
Mỗi mẫu đi qua **cả `k` thành viên** → `k` logit → trung bình softmax khi dự đoán.

In [ ]:
class BatchEnsembleLinear(nn.Module):
    """W dùng chung + adapter rank-1 (r, s) riêng từng thành viên — BatchEnsemble."""
    def __init__(self, in_f, out_f, k):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_f, in_f))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        # khởi tạo r, s bằng dấu ngẫu nhiên ±1 để các thành viên đa dạng
        self.r = nn.Parameter(torch.randint(0, 2, (k, in_f)).float() * 2 - 1)
        self.s = nn.Parameter(torch.randint(0, 2, (k, out_f)).float() * 2 - 1)
        self.bias = nn.Parameter(torch.zeros(k, out_f))

    def forward(self, x):                       # x: (B, k, in)
        x = x * self.r                          # điều biến đầu vào theo từng thành viên
        x = torch.einsum("bki,oi->bko", x, self.weight)   # trọng số dùng chung
        return x * self.s + self.bias           # điều biến đầu ra + bias riêng


class TabM(nn.Module):
    def __init__(self, n_features, n_classes=2, hidden=256, n_blocks=2, k=32, dropout=0.1):
        super().__init__()
        self.k = k
        dims = [n_features] + [hidden] * n_blocks
        self.layers = nn.ModuleList(
            [BatchEnsembleLinear(dims[i], dims[i + 1], k) for i in range(n_blocks)])
        self.drop = nn.Dropout(dropout)
        self.head = BatchEnsembleLinear(hidden, n_classes, k)

    def forward(self, x):                        # x: (B, features)
        x = x.unsqueeze(1).expand(-1, self.k, -1)  # nhân bản cho k thành viên
        for lyr in self.layers:
            x = self.drop(torch.relu(lyr(x)))
        return self.head(x)                       # (B, k, n_classes)

    @torch.no_grad()
    def predict_proba(self, x):                   # trung bình softmax trên k thành viên
        self.eval()
        return self(x).softmax(-1).mean(1)[:, 1].cpu().numpy()

In [ ]:
def fit_tabm(Xtr, ytr, Xval, yval, k=32, epochs=300, patience=25, lr=1e-3, bs=256,
             verbose=False):
    """Train 1 TabM với early stopping theo AUC trên tập validation. Trả model + scaler."""
    sc = StandardScaler().fit(Xtr)
    Xtr_t = torch.tensor(sc.transform(Xtr), dtype=torch.float32, device=DEVICE)
    Xval_t = torch.tensor(sc.transform(Xval), dtype=torch.float32, device=DEVICE)
    ytr_t = torch.tensor(ytr, device=DEVICE)

    model = TabM(Xtr.shape[1], k=k).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()

    best_auc, best_state, wait, history = 0.0, None, 0, []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t), device=DEVICE)
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            logits = model(Xtr_t[idx])                  # (b, k, C)
            b, kk, C = logits.shape
            loss = lossf(logits.reshape(b * kk, C), ytr_t[idx].repeat_interleave(kk))
            opt.zero_grad(); loss.backward(); opt.step()
        auc = roc_auc_score(yval, model.predict_proba(Xval_t))
        history.append(auc)
        if auc > best_auc:
            best_auc, wait = auc, 0
            best_state = {n: p.detach().clone() for n, p in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
        if verbose and ep % 20 == 0:
            print(f"    ep {ep:3d} | val AUC {auc:.4f}")
    model.load_state_dict(best_state)
    return model, sc, best_auc, history


# ---- 5-fold CV cho TabM (early stopping trên 1 phần tách từ train của mỗi fold) ----
tm_auc, tm_f1, tm_acc, last_history = [], [], [], None
t0 = time.time()
for fold, (tr, va) in enumerate(cv.split(X_np, y_np), 1):
    Xtr_in, Xes, ytr_in, yes = train_test_split(
        X_np[tr], y_np[tr], test_size=0.2, stratify=y_np[tr], random_state=SEED)
    model, sc, _, last_history = fit_tabm(Xtr_in, ytr_in, Xes, yes)
    Xva_t = torch.tensor(sc.transform(X_np[va]), dtype=torch.float32, device=DEVICE)
    p = model.predict_proba(Xva_t)
    pred = (p >= 0.5).astype(int)
    tm_auc.append(roc_auc_score(y_np[va], p))
    tm_f1.append(f1_score(y_np[va], pred))
    tm_acc.append(accuracy_score(y_np[va], pred))
    print(f"  fold {fold}: AUC={tm_auc[-1]:.4f}")

print(f"\nTabM | AUC {np.mean(tm_auc):.4f} ± {np.std(tm_auc):.4f} "
      f"| F1 {np.mean(tm_f1):.4f} | Acc {np.mean(tm_acc):.4f} | {time.time()-t0:.1f}s")

leaderboard.append({"Model": "TabM",
                    "Accuracy": np.mean(tm_acc), "Accuracy-Std": np.std(tm_acc),
                    "F1-Score": np.mean(tm_f1), "F1-Std": np.std(tm_f1),
                    "ROC-AUC": np.mean(tm_auc), "ROC-AUC-Std": np.std(tm_auc)})

In [ ]:
# Đường cong AUC theo epoch (của fold cuối) — kiểm tra hội tụ/overfit
plt.figure(figsize=(7, 4))
plt.plot(last_history, marker=".")
plt.xlabel("epoch"); plt.ylabel("validation AUC")
plt.title("TabM — học theo epoch (fold cuối)"); plt.grid(alpha=.3); plt.show()

## 3. Leaderboard tổng hợp

In [ ]:
lb = pd.DataFrame(leaderboard).round(4).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

# Ghép thêm leaderboard baseline của Phase 2 nếu đã lưu trước đó
base_path = DATA_DIR / "baseline_leaderboard.csv"
if base_path.exists():
    lb = pd.concat([pd.read_csv(base_path), lb], ignore_index=True) \
           .sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

out = DATA_DIR / "advanced_models_leaderboard.csv"
lb.to_csv(out, index=False)
print("Đã lưu:", out)
display(lb)

## 4. Submission

Huấn luyện model cuối trên **toàn bộ** train, dự đoán xác suất lớp ASD trên `test`,
xuất file submission (kèm tùy chọn **blend** trung bình hai model).

In [ ]:
ROOT = DATA_DIR.parent.parent           # thư mục gốc dự án
sample = pd.read_csv(ROOT / "data" / "raw" / "sample_submission.csv")

# CatBoost (đã fit ở trên: cb_final)
p_cb = cb_final.predict_proba(X_test.values.astype(np.float32))[:, 1]

# TabM: fit cuối trên toàn train (tách nhỏ để early stopping)
Xtr_in, Xes, ytr_in, yes = train_test_split(X_np, y_np, test_size=0.2,
                                             stratify=y_np, random_state=SEED)
tabm_final, sc_final, _, _ = fit_tabm(Xtr_in, ytr_in, Xes, yes)
Xtest_t = torch.tensor(sc_final.transform(X_test.values.astype(np.float32)),
                       dtype=torch.float32, device=DEVICE)
p_tm = tabm_final.predict_proba(Xtest_t)

id_col = sample.columns[0]
for name, p in [("catboost", p_cb), ("tabm", p_tm), ("blend_cb_tabm", 0.5 * p_cb + 0.5 * p_tm)]:
    sub = pd.DataFrame({id_col: sample[id_col], TARGET: p})
    fname = ROOT / f"submission_{name}.csv"
    sub.to_csv(fname, index=False)
    print("Đã lưu:", fname.name)